# Train your own wake word (openWakeWord) — Odyssync

This notebook makes a small file (`your_phrase.onnx`) that lets Odyssync wake up when you say **your own phrase**, for example "Hey Odyssync".

**How to use it (no coding needed):**
1. **Runtime > Change runtime type > T4 GPU > Save.**
2. Type your phrase in the **Settings** box below.
3. **Runtime > Run all.** Leave the tab open. It takes about **1 hour**.
4. At the end your browser downloads the `.onnx` file. See `tools/wakeword/README.md` for where to put it.

**What happens inside, in plain words:** a computer voice says your phrase thousands of times in hundreds of different voices. We mix those recordings with room echo and background noise, then teach a tiny model to tell "your phrase" apart from everything else (other words, music, TV, talking).

### Sources this notebook follows
- **Official method:** [dscripka/openWakeWord](https://github.com/dscripka/openWakeWord) — `notebooks/automatic_model_training.ipynb` (same data sets, same 2-second clips, same model shape, same "make negatives count more over time" training idea, same Piper voice model `en_US-libritts_r-medium.pt` v2.0.0).
- **Maintained 2026 fixes:** [alfiedennen/openwakeword-colab-2026](https://github.com/alfiedennen/openwakeword-colab-2026) and its fork [pafernanr/openwakeword-colab-2026](https://github.com/pafernanr/openwakeword-colab-2026), shared in the Home Assistant thread ["Colab notebooks to train your custom openWakeWord or microWakeWord"](https://community.home-assistant.io/t/colab-notebooks-to-train-your-custom-openwakeword-or-microwakeword/1026635) (validated May 2026 on Colab's torch 2.11). From these we take: skip the official `train.py` (it breaks on new torch/torchaudio), train with a small hand-written PyTorch loop, bake the sigmoid into the ONNX file, export with `opset 14, dynamo=False`.
- **Our one change:** that same thread reports (Nov 2026) that `piper-phonemize` will not install on Colab's Python 3.13. So we use [rhasspy/piper-sample-generator](https://github.com/rhasspy/piper-sample-generator) **v3.2.0** (Mar 2026), which uses `piper-tts==1.3.0` (its own built-in espeak, works on any Python 3.9+) instead of `piper-phonemize`. We also do the noise/echo mixing with plain numpy so torchaudio/speechbrain/tensorflow are not needed.

**Pinned versions:** piper-sample-generator `v3.2.0` (git tag), piper-tts `1.3.0`, openwakeword `0.6.0`, onnxruntime `1.31.1`, onnx `1.23.2`. torch/numpy/scipy are Colab's own (the 2026 sources report torch 2.11 working).

**Downloads (kept small, about 2.2 GB in total):** Piper voice model (204 MB), room echoes (MIT, ~6 MB), one AudioSet background-noise file (~690 MB), the first part only of openWakeWord's pre-computed "everyday sound" features (~1.2 GB instead of 17 GB), and the false-alarm test set (185 MB).

## Step 1 — Settings
Change the phrase here. **Extra spellings** help the computer voice say an unusual word the way you say it (write it the way it *sounds*). The other numbers are already set for about 1 hour on a free T4 GPU; bigger numbers = better model but slower.

In [ ]:
#@title Settings (edit, then Runtime > Run all)
TARGET_PHRASE = "hey odyssync"  #@param {type:"string"}
EXTRA_SPELLINGS = ["hey oddy sync", "hey odd ee sink"]  #@param {type:"raw"}
POSITIVE_SAMPLES = 3000  #@param {type:"integer"}
NEGATIVE_SAMPLES = 3000  #@param {type:"integer"}
TRAINING_STEPS = 30000  #@param {type:"integer"}
BACKGROUND_FEATURE_ROWS = 400000  #@param {type:"integer"}

import re

TARGET_PHRASE = " ".join(str(TARGET_PHRASE).lower().split())
EXTRA_SPELLINGS = [" ".join(str(s).lower().split()) for s in EXTRA_SPELLINGS if str(s).strip()]
assert TARGET_PHRASE, "Please type a wake phrase in TARGET_PHRASE."
MODEL_NAME = re.sub(r"[^a-z0-9_]", "", TARGET_PHRASE.replace(" ", "_"))
VAL_SAMPLES = max(200, POSITIVE_SAMPLES // 10)
TARGET_FALSE_WAKES_PER_HOUR = 0.5
OUTPUT_FILE = f"/content/{MODEL_NAME}.onnx"

print("Wake phrase:     ", TARGET_PHRASE)
print("Also spoken as:  ", EXTRA_SPELLINGS)
print("Output file name:", MODEL_NAME + ".onnx")

## Step 2 — Check for a GPU
Training needs Google's free graphics card (GPU). If this step stops with a message, do **Runtime > Change runtime type > T4 GPU > Save**, then **Runtime > Run all** again.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("No GPU found.")
    print("Fix: Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save.")
    print("Then: Runtime > Run all.")
    raise RuntimeError("Stopped: please switch to a T4 GPU runtime first (see the message above).")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## Step 3 — Install the tools (about 2 minutes)
Installs the computer voice (Piper), the wake-word library (openWakeWord) and downloads the Piper voice model, which has 904 different speakers.

In [ ]:
!pip install -q piper-tts==1.3.0 onnxruntime==1.31.1 onnx==1.23.2
!pip install -q --no-deps openwakeword==0.6.0

import os
import subprocess

PSG_DIR = "/content/piper-sample-generator"
if not os.path.isdir(f"{PSG_DIR}/piper_train"):
    subprocess.run(["rm", "-rf", PSG_DIR], check=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "v3.2.0",
                    "https://github.com/rhasspy/piper-sample-generator", PSG_DIR], check=True)

VOICE_PT = f"{PSG_DIR}/models/en_US-libritts_r-medium.pt"
VOICE_URL = ("https://github.com/rhasspy/piper-sample-generator/releases/download/"
             "v2.0.0/en_US-libritts_r-medium.pt")
if not os.path.exists(VOICE_PT) or os.path.getsize(VOICE_PT) < 200_000_000:
    subprocess.run(["wget", "-q", "--tries=5", "--timeout=300", "-O", VOICE_PT, VOICE_URL], check=True)
assert os.path.getsize(VOICE_PT) > 200_000_000, "The Piper voice model did not download fully. Run this cell again."
assert os.path.exists(VOICE_PT + ".json"), "The Piper voice settings file is missing."

# openWakeWord's two shared "listening" models (sound -> features). A name that matches no
# ready-made wake word makes download_models fetch only these shared files.
import openwakeword
from openwakeword.utils import download_models
download_models(model_names=["shared-feature-files-only"])
print("Tools ready.")

## Step 4 — Download background sounds (about 5–10 minutes)
- **Room echoes:** recordings of how real rooms bounce sound, so the model works in a bedroom or office.
- **Background noise:** one part of Google's AudioSet (music, traffic, people, pets …).
- **Everyday sound features:** openWakeWord's pre-computed "this is NOT the wake word" examples. We download only the first part (set by `BACKGROUND_FEATURE_ROWS`) instead of the full 17 GB.
- **False-alarm test set:** about 11 hours of ordinary audio, used to count how often the model would wake up by mistake.

In [ ]:
import io
import math
import random
import numpy as np
import requests
import soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, snapshot_download

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
DATA = "/content/data"
os.makedirs(DATA, exist_ok=True)
SR = 16000


def to_16k_mono(x, sr):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(int(sr), SR)
        x = resample_poly(x, SR // g, int(sr) // g).astype(np.float32)
    return x


# (a) Room echoes (MIT impulse responses, already 16 kHz)
snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                  allow_patterns=["16khz/*.wav"], local_dir=f"{DATA}/mit_rirs")
RIRS = []
for name in sorted(os.listdir(f"{DATA}/mit_rirs/16khz")):
    if name.endswith(".wav"):
        x, sr = sf.read(f"{DATA}/mit_rirs/16khz/{name}", dtype="float32")
        RIRS.append(to_16k_mono(x, sr))
print("Room echoes:", len(RIRS))

# (b) Background noise: one AudioSet file (about 490 ten-second clips)
BACKGROUND = []
try:
    import pyarrow.parquet as pq
    bg_file = hf_hub_download("agkphysics/AudioSet", "data/bal_train/09.parquet",
                              repo_type="dataset", local_dir=f"{DATA}/audioset")
    for batch in pq.ParquetFile(bg_file).iter_batches(batch_size=32, columns=["audio"]):
        for item in batch.column("audio").to_pylist():
            try:
                x, sr = sf.read(io.BytesIO(item["bytes"]), dtype="float32")
                x = to_16k_mono(x, sr)
                if len(x) >= SR and np.abs(x).max() > 1e-4:
                    BACKGROUND.append(x)
            except Exception:
                pass
except Exception as error:
    print("Could not use AudioSet background noise:", error)
if not BACKGROUND:
    print("Using made-up noise as background instead (the model will be a little less robust).")
    for _ in range(200):
        white = np.random.randn(10 * SR).astype(np.float32)
        BACKGROUND.append(np.cumsum(white) * 0.01 if random.random() < 0.5 else white)
print("Background clips:", len(BACKGROUND))

# (c) Everyday-sound features (ACAV100M): download only the first BACKGROUND_FEATURE_ROWS rows
ACAV_URL = ("https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
            "openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
ACAV_FILE = f"{DATA}/acav_first_part.npy"
head = io.BytesIO(requests.get(ACAV_URL, headers={"Range": "bytes=0-4095"}, timeout=120).content)
version = np.lib.format.read_magic(head)
if version == (1, 0):
    acav_shape, _, acav_dtype = np.lib.format.read_array_header_1_0(head)
else:
    acav_shape, _, acav_dtype = np.lib.format.read_array_header_2_0(head)
header_bytes = head.tell()
acav_rows = min(int(BACKGROUND_FEATURE_ROWS), acav_shape[0])
row_bytes = int(np.prod(acav_shape[1:])) * acav_dtype.itemsize
want = header_bytes + acav_rows * row_bytes
for attempt in range(6):
    have = os.path.getsize(ACAV_FILE) if os.path.exists(ACAV_FILE) else 0
    if have >= want:
        break
    try:
        with requests.get(ACAV_URL, headers={"Range": f"bytes={have}-{want - 1}"}, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(ACAV_FILE, "ab") as f, tqdm(total=want, initial=have, unit="B", unit_scale=True,
                                                  desc="Everyday sounds") as bar:
                for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                    f.write(chunk)
                    bar.update(len(chunk))
    except Exception as error:
        print("Download interrupted, retrying:", error)
assert os.path.getsize(ACAV_FILE) >= want, "Everyday-sound features did not download. Run this cell again."
ACAV = np.memmap(ACAV_FILE, dtype=acav_dtype, mode="r", offset=header_bytes,
                 shape=(acav_rows,) + tuple(acav_shape[1:]))
print("Everyday-sound examples:", ACAV.shape, f"(about {acav_rows * 1.28 / 3600:.0f} hours)")

# (d) False-alarm test set (~11 hours of features)
val_file = hf_hub_download("davidscripka/openwakeword_features", "validation_set_features.npy",
                           repo_type="dataset", local_dir=DATA)
FP_VAL = np.load(val_file).astype(np.float32)
FP_VAL_HOURS = FP_VAL.shape[0] * 0.08 / 3600
print(f"False-alarm test set: {FP_VAL_HOURS:.1f} hours")

## Step 5 — Make the computer voice say your phrase (about 15–20 minutes)
- **Positive clips:** your phrase (and its extra spellings) in random mixes of the 904 voices, at different speeds.
- **Negative clips:** words that *sound close* but are not your phrase (only part of it, its words in the wrong order, half-words, and common phrases like "hey siri"). These teach the model not to wake up on "almost".

At the end you can **press play** on a few examples to hear how the phrase sounds. If it sounds wrong, add a better extra spelling in Step 1 and run again.

In [ ]:
import sys
import json
import logging
from IPython.display import Audio, display

sys.path.insert(0, PSG_DIR)
import piper_sample_generator.__main__ as psg
logging.getLogger().setLevel(logging.WARNING)

tts = torch.load(VOICE_PT, weights_only=False).eval().cuda()
with open(VOICE_PT + ".json", encoding="utf-8") as f:
    tts_config = json.load(f)
TTS_VOICE = tts_config["espeak"]["voice"]
TTS_RATE = tts_config["audio"]["sample_rate"]
N_SPEAKERS = tts_config["num_speakers"]


def trim_silence(x, rate):
    loud = np.flatnonzero(np.abs(x) > 0.02 * np.abs(x).max())
    pad = int(0.05 * rate)
    return x[max(0, loud[0] - pad): loud[-1] + pad]


def speak(texts, n, desc, batch_size=50):
    """Say random texts from the list n times, each in a random mix of two of the 904 voices."""
    clips = []
    with tqdm(total=n, desc=desc) as bar:
        while len(clips) < n:
            size = min(batch_size, n - len(clips))
            ids = [psg.get_phonemes(TTS_VOICE, tts_config, random.choice(texts)) for _ in range(size)]
            longest = max(len(i) for i in ids)
            ids = [i + [1] * (longest - len(i)) for i in ids]
            speaker_a = torch.randint(0, N_SPEAKERS, (size,))
            speaker_b = torch.randint(0, N_SPEAKERS, (size,))
            with torch.no_grad():
                audio, phoneme_samples = psg.generate_audio(
                    tts, speaker_a, speaker_b, ids,
                    random.choice([0.2, 0.5, 0.8]),       # how much of each voice
                    random.choice([0.5, 0.667, 0.8]),     # voice variation
                    random.choice([0.6, 0.8, 1.0]),       # timing variation
                    random.choice([0.75, 1.0, 1.25]),     # speed
                    None)
            audio = audio.cpu().numpy()[:, 0, :]
            for i in range(size):
                end = int(phoneme_samples[i].flatten().sum().item()) + 1
                x = audio[i, :end]
                if x.size == 0 or np.abs(x).max() < 1e-4:
                    continue
                x = trim_silence(x / np.abs(x).max(), TTS_RATE)
                if len(x) < 0.2 * TTS_RATE:
                    continue
                clips.append(to_16k_mono(x, TTS_RATE))
                bar.update(1)
    return clips


def negative_texts(phrase, spellings):
    common = ["hey", "hello", "hey there", "hey you", "hey siri", "hey google", "okay google", "alexa",
              "hey jarvis", "hey everyone", "hey buddy", "excuse me", "what time is it", "thank you",
              "good morning", "see you soon", "let's go", "never mind", "sounds good", "turn it off",
              "open the door", "is anyone there", "how are you", "one more time"]
    full = {phrase, *spellings}
    texts = set(common)
    for p in full:
        words = p.split()
        for i in range(len(words)):
            for j in range(i + 1, len(words) + 1):
                texts.add(" ".join(words[i:j]))
        if len(words) > 1:
            texts.add(" ".join(reversed(words)))
        for k, word in enumerate(words):
            if len(word) >= 5:
                half = len(word) // 2 + 1
                for piece in (word[:half], word[-half:]):
                    texts.add(piece)
                    texts.add(" ".join(words[:k] + [piece] + words[k + 1:]))
    return sorted(t for t in texts if t and t not in full)


POSITIVE_TEXTS = [TARGET_PHRASE] + [s for s in EXTRA_SPELLINGS if s != TARGET_PHRASE]
NEGATIVE_TEXTS = negative_texts(TARGET_PHRASE, EXTRA_SPELLINGS)
print("Saying:", POSITIVE_TEXTS)
print("Near-misses:", NEGATIVE_TEXTS)

POS_TRAIN = speak(POSITIVE_TEXTS, POSITIVE_SAMPLES, "Your phrase")
POS_VAL = speak(POSITIVE_TEXTS, VAL_SAMPLES, "Your phrase (test)")
NEG_TRAIN = speak(NEGATIVE_TEXTS, NEGATIVE_SAMPLES, "Near-misses")
NEG_VAL = speak(NEGATIVE_TEXTS, VAL_SAMPLES, "Near-misses (test)")
del tts
torch.cuda.empty_cache()

lengths = [len(c) / SR for c in POS_TRAIN]
print(f"Your phrase lasts about {np.median(lengths):.2f} s (the model listens to 1.28 s windows; "
      "phrases up to ~1.5 s work best).")
for clip in POS_TRAIN[:3]:
    display(Audio(clip, rate=SR))

## Step 6 — Add echo and noise, then turn sound into features (about 5–10 minutes)
Each clip is placed in a 2-second window (your phrase ends near the end of the window, like when the app hears you finish speaking), then randomly gets room echo, background noise, hiss and a random volume. Then openWakeWord's own listening models turn each window into the 16 × 96 numbers the wake-word model actually reads.

In [ ]:
from scipy.signal import fftconvolve
from openwakeword.utils import AudioFeatures

CLIP_SAMPLES = 32000  # 2 s, the same window size openWakeWord uses


def place(x, at_end):
    x = x[-CLIP_SAMPLES:]
    out = np.zeros(CLIP_SAMPLES, dtype=np.float32)
    if at_end:
        start = max(0, CLIP_SAMPLES - len(x) - np.random.randint(0, int(0.2 * SR) + 1))
    else:
        start = np.random.randint(0, CLIP_SAMPLES - len(x) + 1)
    out[start:start + len(x)] = x
    return out


def background_window():
    bg = random.choice(BACKGROUND)
    if len(bg) < CLIP_SAMPLES:
        bg = np.tile(bg, CLIP_SAMPLES // len(bg) + 1)
    start = np.random.randint(0, len(bg) - CLIP_SAMPLES + 1)
    return bg[start:start + CLIP_SAMPLES]


def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))


def augment(x):
    speech = x[x != 0]
    speech_rms = rms(speech) if speech.size else 0.0
    if RIRS and np.random.rand() < 0.5:
        wet = fftconvolve(x, random.choice(RIRS))[:CLIP_SAMPLES].astype(np.float32)
        x = wet * (np.abs(x).max() / (np.abs(wet).max() + 1e-9))
        speech_rms = rms(x[np.abs(x) > 1e-4]) if np.any(np.abs(x) > 1e-4) else speech_rms
    if np.random.rand() < 0.75:
        bg = background_window()
        level = (speech_rms or 0.05) / (rms(bg) + 1e-9) / 10 ** (np.random.uniform(-5, 15) / 20)
        x = x + bg * level
    if np.random.rand() < 0.3:
        x = x + np.random.randn(CLIP_SAMPLES).astype(np.float32) * (speech_rms or 0.05) / 10 ** (np.random.uniform(10, 30) / 20)
    peak = np.abs(x).max()
    if peak > 0:
        x = x / peak * np.random.uniform(0.05, 0.9)
    return (x * 32767).astype(np.int16)


def make_windows(clips, at_end, rounds=1):
    out = np.empty((len(clips) * rounds, CLIP_SAMPLES), dtype=np.int16)
    k = 0
    for _ in range(rounds):
        for clip in clips:
            out[k] = augment(place(clip, at_end))
            k += 1
    return out


def background_only(n):
    return np.stack([augment(background_window() * np.random.uniform(0.1, 1.0)) for _ in range(n)])


features = AudioFeatures(inference_framework="onnx", ncpu=os.cpu_count())


def featurize(windows, desc):
    parts = []
    for i in tqdm(range(0, len(windows), 1024), desc=desc):
        f = features.embed_clips(windows[i:i + 1024], batch_size=1024, ncpu=os.cpu_count())
        parts.append(f[:, -16:, :].astype(np.float32))
    return np.concatenate(parts)


X_POS_TRAIN = featurize(make_windows(POS_TRAIN, at_end=True, rounds=2), "Your phrase")
X_POS_VAL = featurize(make_windows(POS_VAL, at_end=True), "Your phrase (test)")
X_NEG_TRAIN = featurize(np.concatenate([make_windows(NEG_TRAIN, at_end=False),
                                        make_windows(NEG_TRAIN[: len(NEG_TRAIN) // 2], at_end=True),
                                        background_only(1000)]), "Near-misses + noise")
X_NEG_VAL = featurize(make_windows(NEG_VAL, at_end=False), "Near-misses (test)")
print("Feature shapes:", X_POS_TRAIN.shape, X_NEG_TRAIN.shape, X_POS_VAL.shape, X_NEG_VAL.shape)

## Step 7 — Train the wake-word model (about 5–10 minutes)
The model sees your phrase, near-misses and lots of everyday sound. Over time we make every false wake-up "cost" more (up to 1500×), so it learns to stay quiet unless it is sure. Every so often we test it:
- **recall** = how many of your test phrases it catches (higher is better),
- **false wakes / hour** = how often it would wake up by mistake on 11 hours of normal audio (lower is better).

At the end we keep the version that catches the most phrases while waking up by mistake at most about 0.5 times an hour.

In [ ]:
import copy
import time
import torch.nn as nn

DEVICE = "cuda"


class WakeWordNet(nn.Module):
    """Same shape as openWakeWord's default 'dnn' model (1 block, 128 wide)."""

    def __init__(self, frames=16, dims=96, width=128):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(frames * dims, width)
        self.layernorm1 = nn.LayerNorm(width)
        self.relu1 = nn.ReLU()
        self.block = nn.Sequential(nn.Linear(width, width), nn.LayerNorm(width), nn.ReLU())
        self.last_layer = nn.Linear(width, 1)

    def forward(self, x):
        x = self.relu1(self.layernorm1(self.layer1(self.flatten(x))))
        return self.last_layer(self.block(x)).squeeze(-1)  # raw score; sigmoid is added on export


pos_train = torch.from_numpy(X_POS_TRAIN).to(DEVICE)
neg_train = torch.from_numpy(X_NEG_TRAIN).to(DEVICE)
pos_val = torch.from_numpy(X_POS_VAL).to(DEVICE)
neg_val = torch.from_numpy(X_NEG_VAL).to(DEVICE)
acav = torch.from_numpy(np.ascontiguousarray(ACAV)).to(DEVICE)  # float16, stays on the GPU
fp_val = torch.from_numpy(FP_VAL).to(DEVICE)

model = WakeWordNet().to(DEVICE)
bce = nn.BCEWithLogitsLoss(reduction="none")
B_POS, B_NEG, B_ACAV = 128, 128, 512
history, checkpoints = [], []


def make_batch():
    p = pos_train[torch.randint(0, len(pos_train), (B_POS,), device=DEVICE)]
    n = neg_train[torch.randint(0, len(neg_train), (B_NEG,), device=DEVICE)]
    a = acav[torch.randint(0, len(acav), (B_ACAV,), device=DEVICE)].float()
    y = torch.cat([torch.ones(B_POS, device=DEVICE), torch.zeros(B_NEG + B_ACAV, device=DEVICE)])
    return torch.cat([p, n, a]), y


@torch.no_grad()
def false_wakes_per_hour(threshold=0.5, chunk=65536):
    wakes, previous = 0, False
    for i in range(0, fp_val.shape[0] - 15, chunk):
        windows = fp_val[i:i + chunk + 15].unfold(0, 16, 1).permute(0, 2, 1)
        above = torch.sigmoid(model(windows)) >= threshold
        starts = above[1:] & ~above[:-1]
        wakes += int(starts.sum()) + int(bool(above[0]) and not previous)
        previous = bool(above[-1])
    return wakes / FP_VAL_HOURS


@torch.no_grad()
def evaluate(step):
    model.eval()
    recall = (torch.sigmoid(model(pos_val)) >= 0.5).float().mean().item()
    near_miss_ok = (torch.sigmoid(model(neg_val)) < 0.5).float().mean().item()
    fph = false_wakes_per_hour()
    model.train()
    history.append((step, recall, near_miss_ok, fph))
    checkpoints.append((recall, fph, copy.deepcopy(model.state_dict())))
    print(f"step {step:>6}: recall {recall:.3f} | near-misses ignored {near_miss_ok:.3f} | "
          f"false wakes/hour {fph:.2f}", flush=True)


def train(steps, top_lr, weight_from, weight_to, check_every):
    optimizer = torch.optim.Adam(model.parameters(), lr=top_lr)
    neg_weights = np.linspace(weight_from, weight_to, steps)
    warmup, hold = max(1, steps // 5), steps // 3
    for step in range(steps):
        if step < warmup:
            lr = top_lr * (step + 1) / warmup
        elif step < warmup + hold:
            lr = top_lr
        else:
            lr = top_lr * 0.5 * (1 + math.cos(math.pi * (step - warmup - hold) / max(1, steps - warmup - hold)))
        for group in optimizer.param_groups:
            group["lr"] = lr
        x, y = make_batch()
        logits = model(x)
        p = torch.sigmoid(logits).detach()
        keep = ((y == 0) & (p >= 0.001)) | ((y == 1) & (p < 0.999))  # only learn from examples it still gets wrong
        if keep.any():
            weight = torch.where(y[keep] > 0.5, torch.ones_like(y[keep]),
                                 torch.full_like(y[keep], float(neg_weights[step])))
            loss = (bce(logits[keep], y[keep]) * weight).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if (step + 1) % check_every == 0:
            evaluate(step + 1)


started = time.time()
check_every = max(500, TRAINING_STEPS // 25)
print("Main training")
train(TRAINING_STEPS, 1e-4, 1.0, 1500.0, check_every)
print("Fine-tuning")
train(max(2000, TRAINING_STEPS // 10), 1e-5, 1500.0, 1500.0, max(500, check_every // 2))
print(f"Training took {(time.time() - started) / 60:.1f} minutes")

good = [c for c in checkpoints if c[1] <= TARGET_FALSE_WAKES_PER_HOUR]
best = max(good, key=lambda c: c[0]) if good else min(checkpoints, key=lambda c: (c[1], -c[0]))
model.load_state_dict(best[2])
model.eval()
print(f"Chosen version: recall {best[0]:.3f}, false wakes/hour {best[1]:.2f}")
if best[0] < 0.5:
    print("Recall is low. Try more POSITIVE_SAMPLES / TRAINING_STEPS, or better EXTRA_SPELLINGS.")

## Step 8 — Save as an ONNX file and test it the way the app will
We save the model, then load it with openWakeWord exactly like Odyssync does (80 ms chunks of 16 kHz audio, threshold 0.5) and play some test clips through it.

In [ ]:
import onnxruntime as ort
from openwakeword.model import Model


class WithSigmoid(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x):
        return torch.sigmoid(self.base(x)).unsqueeze(-1)  # shape [batch, 1], score 0..1


export_model = WithSigmoid(copy.deepcopy(model).cpu()).eval()
dummy = torch.zeros(1, 16, 96)
export_args = dict(input_names=["input"], output_names=["score"],
                   dynamic_axes={"input": {0: "batch"}, "score": {0: "batch"}}, opset_version=14)
try:
    torch.onnx.export(export_model, dummy, OUTPUT_FILE, dynamo=False, **export_args)
except TypeError:  # older torch without the dynamo switch
    torch.onnx.export(export_model, dummy, OUTPUT_FILE, **export_args)
print("Saved", OUTPUT_FILE, f"({os.path.getsize(OUTPUT_FILE) / 1e3:.0f} KB)")

# 1) The ONNX file gives the same scores as the trained model
session = ort.InferenceSession(OUTPUT_FILE, providers=["CPUExecutionProvider"])
sample = X_POS_VAL[:256]
onnx_scores = session.run(None, {session.get_inputs()[0].name: sample})[0].reshape(-1)
with torch.no_grad():
    torch_scores = export_model(torch.from_numpy(sample)).numpy().reshape(-1)
assert np.max(np.abs(onnx_scores - torch_scores)) < 1e-4, "ONNX export does not match the trained model."

# 2) openWakeWord loads it and detects it in a live-like stream
oww = Model(wakeword_models=[OUTPUT_FILE], inference_framework="onnx")


def best_stream_score(clip_int16):
    oww.reset()
    audio = np.concatenate([np.zeros(SR, np.int16), clip_int16, np.zeros(SR // 2, np.int16)])
    best = 0.0
    for i in range(0, len(audio) - 1279, 1280):
        best = max(best, float(oww.predict(audio[i:i + 1280])[MODEL_NAME]))
    return best


test_pos = make_windows(POS_VAL[:40], at_end=True)
test_neg = make_windows(NEG_VAL[:40], at_end=False)
caught = sum(best_stream_score(c) >= 0.5 for c in test_pos)
fooled = sum(best_stream_score(c) >= 0.5 for c in test_neg)
print(f"Streaming test: caught {caught}/{len(test_pos)} of your phrases, "
      f"woke on {fooled}/{len(test_neg)} near-misses.")

## Step 9 — Download your wake word
Your browser downloads the file (allow downloads if it asks). If nothing happens, open the **folder icon** on the left and right-click the `.onnx` file > **Download**.

**Where to put it** (details in `tools/wakeword/README.md`):
- In the Odyssync repo: `sidecar/models/` (ships inside the app — use this for Hey Odyssync), or
- On a Windows laptop: `%APPDATA%\Odyssync\wakewords\` (Control Center > wake word > "Add your own…" opens this folder).

It then shows up in the Control Center's wake-word list, named from the file name (`hey_odyssync.onnx` → "Hey Odyssync").

In [ ]:
from google.colab import files

print("Downloading", os.path.basename(OUTPUT_FILE))
files.download(OUTPUT_FILE)